# 教程 0：十分钟上手 PLLM

PLLM 让你在 **LLM 推理的同时**运行自己的张量数学：观测 hidden、在 decode 途中
训练一个小头、给某些请求挂上 LoRA、把信息从一个请求传给下一个请求……
而且不改变 vLLM 的推理语义，也不需要你碰调度器、KV cache 或 CUDA 流。

你只需要会 PyTorch。整个框架对用户只有三个概念：

1. **Workload** —— 一个 Python 类，用类型标注声明：*算什么数学、读哪些张量、
   用什么状态、在什么时机运行*。
2. **签名编译** —— `make_llm` 把你的声明编译成一份由 Rust 签名的执行计划；
   形状、dtype、依赖、效果在这里被一次性检查固定。
3. **托管运行时** —— 状态版本、隔离、发布、回收全部由运行时按签名计划执行；
   你的数学代码只负责算。

本教程先把最基础的链路跑通；后续每一篇用一个真实场景教你一组概念。

## 第一次生成

`make_llm` 不传 workload 时就是普通 vLLM 推理。`ModelConfig` 的位置参数依次是
模型、dtype、最大长度、并发数、批 token 数、显存占比。

In [1]:
import os
os.environ.setdefault("VLLM_ENABLE_V1_MULTIPROCESSING", "0")

import pllm
from vllm import SamplingParams
from pllm.config import VllmDType, ModelConfig

llm = pllm.make_llm(config=ModelConfig(
    "Qwen/Qwen2.5-0.5B-Instruct", VllmDType.BFLOAT16,
    64,   # max_model_len
    2,    # max_num_seqs
    128,  # max_num_batched_tokens
    0.2,  # gpu_memory_utilization
    seed=7))

outputs = llm.generate(
    ["The sky is", "Water freezes at"],
    SamplingParams(max_tokens=8, temperature=0.0, ignore_eos=True),
    use_tqdm=False)
for output in outputs:
    print(repr(output.outputs[0].text))

/workspace/pllm/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-09-10 12:25:45,095	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 09-10 12:25:45 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'seed': 7, 'max_model_len': 64, 'distributed_executor_backend': 'uni', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'max_num_batched_tokens': 128, 'max_num_seqs': 2, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


WARNING 09-10 12:25:46 [arg_utils.py:1678] The global random seed is set to 7. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 09-10 12:25:47 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 09-10 12:25:47 [model.py:1883] Using max model len 64


... (26 outputs left out)

INFO 09-10 12:25:55 [jit_monitor.py:79] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.


INFO 09-10 12:25:55 [core.py:355] init engine (profile, create kv cache, warmup model) took 1.60 s


INFO 09-10 12:25:59 [hf.py:540] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


' a beautiful place. It is a place'
' 0 degrees Celsius and boils at '


## 挂上你的第一个 workload

Workoad 是一个继承 `pllm.Workload` 的类。下面这个最小 workload 只做一件事：
每一步、每个请求，把第 0 层 hidden 的均值**导出**到宿主机。

先不用纠结细节 —— 教程 1 会逐步拆开。现在只要注意三件事：

- `@pllm.on(pllm.Phase.OBSERVE)` 声明**时机**（采样前的观测窗口）；
- 参数上的 `Annotated[torch.Tensor, pllm.Hidden(0)]` 声明**输入来源**（第 0 层 hidden）；
- `@pllm.export(...)` 声明**效果**（把结果有界地复制到宿主机 trace）。

整个类里没有一行调度、锁或设备代码。

In [2]:
from typing import Annotated
import torch


class PeekHidden(pllm.Workload):
    """每步导出第 0 层 hidden 的行均值。"""

    @pllm.on(pllm.Phase.OBSERVE)
    @pllm.export(name="hidden_mean", every=1, max_rows=4, max_cols=896)
    def observe(self, hidden: Annotated[torch.Tensor, pllm.Hidden(0)]) -> torch.Tensor:
        return hidden.float().mean(dim=-1, keepdim=True)

## 编译、运行、读结果

`make_llm(workload=...)` 会编译并签名你的声明。导出的事件躺在有界 trace 里，
`trace_snapshot()` 随时可取；每个事件带有 step_id、请求身份和 CPU 张量。
`worker_state().diagnostics` 是执行面的体检报告 —— `fallback_count == 0`
表示一切都在签名原语内运行。

In [3]:
import gc

# 先释放上一个引擎，再给 workload 引擎让出显存
llm.llm_engine.engine_core.shutdown()
del llm
gc.collect()
torch.cuda.empty_cache()

peek = pllm.make_llm(workload=PeekHidden(), config=ModelConfig(
    "Qwen/Qwen2.5-0.5B-Instruct", VllmDType.BFLOAT16, 64, 2, 128, 0.2, seed=7))
try:
    peek.generate(["The sky is", "Water freezes at"],
        SamplingParams(max_tokens=4, temperature=0.0, ignore_eos=True), use_tqdm=False)
    for event in peek.trace_snapshot().events:
        if event.tensor_name == "hidden_mean":
            print(f"step {event.step_id}: 请求 "
                  f"{[i.prompt_idx for i in event.identities]} 的均值 "
                  f"{event.tensor().flatten().tolist()}")
    print("diagnostics:", peek.worker_state().diagnostics)
finally:
    peek.close()

INFO 09-10 12:26:00 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'seed': 7, 'max_model_len': 64, 'distributed_executor_backend': 'uni', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'max_num_batched_tokens': 128, 'max_num_seqs': 2, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'worker_cls': 'pllm.worker.GPUWorker', 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


WARNING 09-10 12:26:00 [arg_utils.py:1678] The global random seed is set to 7. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 09-10 12:26:01 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 09-10 12:26:01 [model.py:1883] Using max model len 64


INFO 09-10 12:26:01 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])


INFO 09-10 12:26:02 [core.py:121] Initializing a V1 LLM engine (v0.27.1) with config: model='Qwen/Qwen2.5-0.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-0.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=64, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collec

... (9 outputs left out)

INFO 09-10 12:26:04 [kv_cache_utils.py:2236] Maximum concurrency for 64 tokens per request: 5073.25x


INFO 09-10 12:26:04 [gpu_worker.py:789] Free memory on device (11.64/23.48 GiB) on startup. Desired GPU memory utilization is (0.2, 4.7 GiB). Actual usage is 0.98 GiB for consumed memory (weights + non-torch), 0.0 GiB for peak activation, and 0.0 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=3832543232` (3.57 GiB) to fit into requested memory, or `--kv-cache-memory=11282675712` (10.51 GiB) to fully utilize gpu memory. Current kv cache memory in use is 3.72 GiB.


INFO 09-10 12:26:04 [core.py:355] init engine (profile, create kv cache, warmup model) took 0.90 s


step 1: 请求 [0, 1] 的均值 [0.0001381508045597002, -0.0018154947319999337]
step 2: 请求 [0, 1] 的均值 [0.0024056904949247837, 0.006180447060614824]
step 3: 请求 [0, 1] 的均值 [0.0032387205865234137, 0.000969733577221632]
diagnostics: StateDiagnostics(active_lineages=0, inference_pins=0, loss_count=0, update_count=0, trained_rows=0, completed_sampler_pins=3, pending_state_releases=0, fallback_count=0, state_usage=PageBundleUsage(reserved_bytes=0, logical_live_bytes=0, allocated_payload_bytes=0, reserved_padding_bytes=0, free_field_pages=0, live_field_pages=0, private_field_pages=0, retiring_field_pages=0, private_versions=0, retiring_versions=0, pinned_versions=0, pending_receipts=0, admission_capacity_failures=0))


## 你现在已经知道的

- `make_llm(config=...)` 是 vLLM；`make_llm(workload=..., config=...)` 是 vLLM + 你的数学。
- workload 用标注声明**时机、输入、效果**；框架负责签名与执行。
- 结果从 trace 里读，健康状况看 diagnostics。

接下来：

- [教程 1：观测与 SAE](01_observe_sae.ipynb) —— 冻结状态、多层观测、非阻塞落盘；
- [教程 2：写自己的训练头](02_custom_head.ipynb) —— 逐步蒸馏、采样引导、CNN；
- [教程 3：按请求路由的 LoRA](03_request_lora.ipynb) —— 请求隔离、route 共享、混合与调度；
- [教程 4：三种循环](04_loops.ipynb) —— 层循环、步循环、会话循环；
- [教程 5：decode 时在线训练](05_decode_training.ipynb) —— 跨步与跨请求的训练目标。